# 04c — Complete-case sensitivity network

Predefined sensitivity analysis evaluating the influence of missing-data handling (Methods 2.6): the 29 primary biomarkers are reconstructed on participants with no missing values (no imputation), using the identical EBIC(gamma=0.5)+density<=0.20 selection rule and centrality/bridge formulas as the primary network, then compared edge-by-edge and rank-by-rank against the primary (imputed) network.

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT / "outputs").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if _in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import json
import numpy as np, pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

fd = pd.read_csv(PROJECT_ROOT / 'outputs' / '01_ingest_harmonize' / 'feature_dictionary_v2.csv')
primary = fd.loc[fd.included_primary, 'feature'].tolist()
cols = primary
p = len(cols)

Y = pd.read_csv(PROJECT_ROOT / 'outputs' / '02_primary_network_dataset' / 'primary_direct_features_unimputed.csv')
Z = pd.read_csv(PROJECT_ROOT / 'outputs' / '02_primary_network_dataset' / 'primary_direct_features_z.csv')

primary_summary = json.loads((PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso' / 'primary_graphical_lasso_model_summary.json').read_text())
edges = pd.read_csv(PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso' / 'primary_graphical_lasso_partial_edges.csv')
selected_alpha = primary_summary['selected_alpha']
metrics, _ = graph_metrics(edges, {f: DOM[f] for f in cols})

CC = Y.dropna(subset=cols).reset_index(drop=True)
n_complete = len(CC)
Zcc = pd.DataFrame(StandardScaler().fit_transform(CC[cols]), columns=cols)
model_cc, alpha_table_cc, selected_alpha_cc = select_glasso_ebic(Zcc, max_density=0.20)

out_dir = PROJECT_ROOT / 'outputs' / '04c_complete_case_network'
out_dir.mkdir(parents=True, exist_ok=True)
alpha_table_cc.to_csv(out_dir / 'complete_case_alpha_selection_ebic.csv', index=False)

prec_cc = model_cc.precision_
D_cc = np.sqrt(np.diag(prec_cc))
P_cc = -prec_cc / np.outer(D_cc, D_cc)
np.fill_diagonal(P_cc, 1)
edges_cc = edge_df_from_partial(P_cc, cols)
edges_cc['source_domain'] = edges_cc.source.map(DOM)
edges_cc['target_domain'] = edges_cc.target.map(DOM)
edges_cc['cross_domain'] = edges_cc.source_domain != edges_cc.target_domain
edges_cc.to_csv(out_dir / 'complete_case_partial_edges.csv', index=False)
pd.DataFrame(P_cc, index=cols, columns=cols).to_csv(out_dir / 'complete_case_partial_correlation_matrix.csv')

summary_cc = {
    'n_participants_complete_case': int(n_complete),
    'n_participants_primary_imputed': int(len(Z)),
    'pct_of_primary_cohort_retained': round(100 * n_complete / len(Z), 1),
    'n_primary_direct_biomarkers': int(p),
    'selection_method': 'EBIC gamma=0.50 with pre-specified sparse interpretability constraint density <= 0.20 (identical rule to primary network)',
    'selected_alpha': float(selected_alpha_cc),
    'n_edges': int(len(edges_cc)),
    'network_density': float(len(edges_cc) / (p * (p - 1) / 2)),
    'n_cross_domain_edges': int(edges_cc.cross_domain.sum()),
    'note': 'Complete-case sensitivity network: rows with any missing value among the 29 primary biomarkers were excluded (no imputation). Same preprocessing, z-scoring, Graphical LASSO EBIC(gamma=0.5)+density<=0.20 selection rule, and centrality/bridge-centrality formulas as the primary network.',
}
(out_dir / 'complete_case_model_summary.json').write_text(json.dumps(summary_cc, indent=2), encoding='utf-8')

metrics_cc, _ = graph_metrics(edges_cc, {f: DOM[f] for f in cols})
metrics_cc.to_csv(out_dir / 'Table_complete_case_centrality_bridge_metrics.csv', index=False)
metrics_cc.head(10).to_csv(out_dir / 'top_bridge_biomarkers_complete_case.csv', index=False)
metrics_cc.sort_values('strength', ascending=False).head(10).to_csv(out_dir / 'top_central_biomarkers_complete_case.csv', index=False)

print(json.dumps(summary_cc, indent=2))


### Comparison against the primary network

In [ ]:
es_primary = set(tuple(sorted((r.source, r.target))) for r in edges.itertuples())
es_cc = set(tuple(sorted((r.source, r.target))) for r in edges_cc.itertuples())
shared_cc = es_primary & es_cc

merged_cc = metrics[['feature', 'bridge_strength', 'strength']].merge(
    metrics_cc[['feature', 'bridge_strength', 'strength']], on='feature', suffixes=('_primary', '_cc'))
top8p = set(metrics.sort_values('bridge_strength', ascending=False).head(8).feature)
top8c = set(metrics_cc.sort_values('bridge_strength', ascending=False).head(8).feature)

comparison_cc = {
    'primary_n': int(len(Z)), 'complete_case_n': int(n_complete),
    'primary_alpha': float(selected_alpha), 'complete_case_alpha': float(selected_alpha_cc),
    'primary_n_edges': int(len(edges)), 'complete_case_n_edges': int(len(edges_cc)),
    'primary_density': float(len(edges) / (p * (p - 1) / 2)),
    'complete_case_density': float(len(edges_cc) / (p * (p - 1) / 2)),
    'n_shared_edges': len(shared_cc),
    'jaccard_edge_similarity': len(shared_cc) / len(es_primary | es_cc) if (es_primary | es_cc) else None,
    'pct_primary_edges_preserved_in_complete_case': len(shared_cc) / len(es_primary) * 100 if es_primary else None,
    'bridge_strength_spearman_correlation': float(merged_cc.bridge_strength_primary.corr(merged_cc.bridge_strength_cc, method='spearman')),
    'node_strength_spearman_correlation': float(merged_cc.strength_primary.corr(merged_cc.strength_cc, method='spearman')),
    'top8_bridge_biomarkers_primary': sorted(top8p),
    'top8_bridge_biomarkers_complete_case': sorted(top8c),
    'top8_overlap': sorted(top8p & top8c), 'top8_overlap_count': len(top8p & top8c),
    'top8_only_in_primary': sorted(top8p - top8c), 'top8_only_in_complete_case': sorted(top8c - top8p),
}
(out_dir / 'complete_case_vs_primary_comparison.json').write_text(json.dumps(comparison_cc, indent=2), encoding='utf-8')
print(json.dumps(comparison_cc, indent=2))


### Figures

In [ ]:
plt.figure(figsize=(11, 9))
Gcc = nx.Graph()
for f in cols:
    Gcc.add_node(f, domain=DOM[f])
for _, r in edges_cc.iterrows():
    Gcc.add_edge(r.source, r.target, weight=r.weight, abs_weight=r.abs_weight)
pos = nx.spring_layout(Gcc, seed=SEED, k=0.55)
widths = [max(0.5, 4 * Gcc[u][v]['abs_weight']) for u, v in Gcc.edges]
nx.draw_networkx_edges(Gcc, pos, width=widths, alpha=.45)
nx.draw_networkx_nodes(Gcc, pos, node_size=650)
nx.draw_networkx_labels(Gcc, pos, font_size=8)
plt.axis('off'); plt.tight_layout()
plt.savefig(out_dir / 'Figure_complete_case_partial_network.png', dpi=200)
plt.show()

for col, fn, title in [
    ('bridge_strength', 'Figure_top_bridge_strength_complete_case.png', 'Top bridge biomarkers (complete-case)'),
    ('strength', 'Figure_top_node_strength_complete_case.png', 'Top central biomarkers (complete-case)'),
]:
    top = metrics_cc.sort_values(col, ascending=False).head(12).iloc[::-1]
    plt.figure(figsize=(8, 5)); plt.barh(top.feature, top[col]); plt.title(title); plt.tight_layout()
    plt.savefig(out_dir / fn, dpi=200); plt.show()
